# Central limma correction

Each omics layer is corrected separately with `limma::removeBatchEffect` using the design `~ condition` with `batch` passed as the technical factor and `D5/F7/M8` as donor covariates (`D6` = reference donor, matching Quartet figshare 22188349).

**Reads:** `before/<Modality>/central_intensities_log_UNION.tsv` + `metadata.tsv` (written by `02_prepare_RBE_inputs.ipynb`).  

**Writes:** `after/<Modality>/intensities_log_Rcorrected_UNION.tsv` + `after/<Modality>/metadata.tsv`.  

**Prerequisite for:** `04_run_fedrbe.ipynb` uses the same reference-batch ordering to achieve exact numerical alignment with FedRBE. The combined k-means input matrices are built by `evaluation_clusterization_after_correction/real_datasets/multiomics/00_build_kmeans_matrices.ipynb`.


In [1]:
library(tidyverse)
library(limma)

Warning message:
“package ‘dplyr’ was built under R version 4.5.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
library(tidyverse)
library(limma)
source(file.path(if (dir.exists("figshare_data")) "." else "evaluation_data/multiomics",
                 "..", "..", "evaluation_utils", "plots_eda.R"))

WORKDIR <- if (dir.exists("figshare_data")) "." else "evaluation_data/multiomics"
BEFORE_DIR <- file.path(WORKDIR, "before")
AFTER_DIR <- file.path(WORKDIR, "after")
PLOTS_DIR <- file.path(WORKDIR, "plots")

dir.create(AFTER_DIR, showWarnings = FALSE, recursive = TRUE)
dir.create(PLOTS_DIR, showWarnings = FALSE, recursive = TRUE)

MODALITIES <- c("Transcriptomics", "Proteomics", "Metabolomics")
# Reference donor first; D6 becomes the limma intercept condition.
DONOR_LEVELS <- c("D6", "D5", "F7", "M8")
DONOR_REFERENCE <- DONOR_LEVELS[1]
COVARIATE_LEVELS <- DONOR_LEVELS[-1]
# Toggle: include the synthetic client_04_L03_L14 (L03 + L14 fold-in).
# Default FALSE -- must match the toggles in 01_preprocess_eda.ipynb and
# 02_prepare_RBE_inputs.ipynb. Used here only as documentation; correction
# operates on whichever clients are present in before/<Modality>/metadata.tsv.
INCLUDE_CLIENT_04 <- FALSE
ALL_CLIENT_NAMES <- c("client_01_L01", "client_02_L02",
                     "client_03_L05_L04", "client_04_L03_L14")
OPTIONAL_CLIENTS <- c("client_04_L03_L14")
CLIENT_NAMES <- if (INCLUDE_CLIENT_04) {
  ALL_CLIENT_NAMES
} else {
  setdiff(ALL_CLIENT_NAMES, OPTIONAL_CLIENTS)
}
N_DONORS <- length(DONOR_LEVELS)



Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine


Loading required package: viridisLite



## Helper functions

`plot_diagnostics` uses `pca_plot()`, `boxplot_plot()`, and `plotIntensityDensity()` from `evaluation_utils/plots_eda.R`, combined into one patchwork panel per (modality, state).


In [3]:
read_feature_matrix <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)
  first_col <- names(df)[1]
  mat <- df %>%
    column_to_rownames(first_col) %>%
    as.data.frame(check.names = FALSE)
  mat[] <- lapply(mat, as.numeric)
  as.matrix(mat)
}

write_feature_matrix <- function(expr, path) {
  out <- as.data.frame(expr, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out, file = path, sep = "\t", quote = TRUE, row.names = FALSE,
              col.names = TRUE, na = "NA")
}

load_prepared_modality <- function(modality) {
  modality_dir <- file.path(BEFORE_DIR, modality)
  expr <- read_feature_matrix(file.path(modality_dir, "central_intensities_log_UNION.tsv"))
  metadata <- readr::read_tsv(file.path(modality_dir, "metadata.tsv"), show_col_types = FALSE) %>%
    mutate(
      condition = factor(condition, levels = DONOR_LEVELS),
      batch = factor(batch),
      rep = as.integer(rep)
    )

  missing <- setdiff(metadata$file, colnames(expr))
  if (length(missing) > 0) stop(modality, ": missing matrix samples: ", paste(head(missing), collapse = ", "), call. = FALSE)
  expr <- expr[, metadata$file, drop = FALSE]
  list(expr = expr, metadata = metadata)
}

# Validates balanced donor/batch structure and full design-matrix rank.
assert_limma_design <- function(metadata, modality) {
  donor_counts <- metadata %>% count(condition) %>% pull(n)
  if (n_distinct(donor_counts) != 1) {
    stop(modality, ": unbalanced donors (", paste(donor_counts, collapse = "/"), ").", call. = FALSE)
  }
  cell_counts <- metadata %>% count(batch_code, batch, condition) %>% pull(n)
  if (n_distinct(cell_counts) != 1) {
    stop(modality, ": unequal replicates per (batch, condition) cell.", call. = FALSE)
  }

  full_design <- model.matrix(~ condition + batch, data = metadata)
  rank <- qr(full_design)$rank
  if (rank != ncol(full_design)) {
    stop(modality, ": limma design is not full rank: ", rank, " < ", ncol(full_design), call. = FALSE)
  }
  invisible(TRUE)
}

# Missing cells are expected for client-wise soft filtering. Correction must
# preserve that mask exactly and keep every observed cell finite.
validate_corrected_matrix <- function(expr, corrected, modality) {
  input_na     <- is.na(expr)
  corrected_m  <- as.matrix(corrected)
  corrected_na <- is.na(corrected_m)

  if (!identical(dim(input_na), dim(corrected_na)))
    stop(modality, ": corrected matrix dimensions changed.", call. = FALSE)
  if (any(corrected_na & !input_na))
    stop(modality, ": correction introduced new NA values.", call. = FALSE)
  if (any(input_na & !corrected_na))
    stop(modality, ": correction filled expected NA values.", call. = FALSE)
  if (any(!corrected_na & !is.finite(corrected_m)))
    stop(modality, ": corrected observed cells contain non-finite values.", call. = FALSE)
  if (any(corrected_na))
    cat(modality, ": preserved", sum(corrected_na), "expected NA cells after correction\n")
  invisible(TRUE)
}

# Saves a diagnostic panel per (modality, state): PCA by batch, PCA by donor,
# sample boxplot coloured by batch, density by batch — combined into one
# patchwork file via plots_eda.R utilities.
plot_diagnostics <- function(expr, metadata, modality, state_label) {
  pca_batch <- pca_plot(expr, metadata,
    title = paste(modality, state_label, "by batch"),
    quantitative_col_name = "file", col_col = "batch_code", shape_col = "condition",
    point_size = 2.5)

  pca_condition <- pca_plot(expr, metadata,
    title = paste(modality, state_label, "by donor"),
    quantitative_col_name = "file", col_col = "condition", shape_col = "batch_code",
    point_size = 2.5)

  box_by_batch <- boxplot_plot(expr, metadata,
    quantitativeColumnName = "file", color_col = "batch_code",
    title = paste(modality, state_label, "batch distributions"),
    remove_xnames = TRUE)

  density_by_batch <- plotIntensityDensity(expr, metadata,
    quantitativeColumnName = "file", colorColumnName = "batch_code",
    title = paste(modality, state_label))

  layout <- (pca_batch | pca_condition) / box_by_batch / density_by_batch
  out_path <- file.path(PLOTS_DIR, paste0(modality, "_limma_", state_label, "_diagnostics.png"))
  ggsave(out_path, layout, width = 14, height = 16, bg = "white")

  tibble(modality = modality, state = state_label, pca_features = nrow(na.omit(expr)))
}


## Correct each modality independently

`batch` is the technical factor removed by limma. Donor condition stays in the design so D5/D6/F7/M8 signal is preserved for clustering.

In [4]:
diagnostic_summaries <- list()

correction_summaries <- purrr::map_dfr(MODALITIES, function(modality) {
  message("Correcting ", modality)
  prepared <- load_prepared_modality(modality)
  expr <- prepared$expr
  metadata <- prepared$metadata
  assert_limma_design(metadata, modality)

  diagnostic_summaries[[paste(modality, "before", sep = "_")]] <<- plot_diagnostics(expr, metadata, modality, "before")

  # Align central limma's sum-contrast reference with FedRBE's reference
  # (last batch of the last client group). removeBatchEffect() forces
  # contr.sum coding, whose all-minus-one reference row is the LAST factor
  # level. This matters for features missing an entire client: using the
  # wrong level leaves the corrections equal only up to a per-row constant.
  ordered_batches <- metadata %>%
    distinct(batch_code, batch) %>%
    arrange(batch_code) %>%
    pull(batch) %>%
    as.character()
  fedrbe_reference_batch <- tail(ordered_batches, 1)
  batch_levels <- c(setdiff(ordered_batches, fedrbe_reference_batch), fedrbe_reference_batch)
  metadata$batch <- factor(as.character(metadata$batch), levels = batch_levels)

  design <- model.matrix(~ condition, data = metadata)
  colnames(design) <- c("Intercept", COVARIATE_LEVELS)
  corrected <- limma::removeBatchEffect(as.matrix(expr), batch = metadata$batch, design = design) %>%
    as.data.frame(check.names = FALSE)

  validate_corrected_matrix(expr, corrected, modality)

  modality_after_dir <- file.path(AFTER_DIR, modality)
  dir.create(modality_after_dir, showWarnings = FALSE, recursive = TRUE)
  write_feature_matrix(corrected, file.path(modality_after_dir, "intensities_log_Rcorrected_UNION.tsv"))
  write.table(metadata, file = file.path(modality_after_dir, "metadata.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

  diagnostic_summaries[[paste(modality, "after", sep = "_")]] <<- plot_diagnostics(as.matrix(corrected), metadata, modality, "after")

  tibble(
    modality = modality,
    features_before = nrow(expr),
    features_after = nrow(corrected),
    samples = ncol(corrected),
    input_missing_cells = sum(is.na(expr)),
    corrected_missing_cells = sum(is.na(corrected)),
    corrected_rows_with_any_na = sum(rowSums(is.na(corrected)) > 0),
    batches = n_distinct(metadata$batch),
    design_rank = qr(model.matrix(~ condition + batch, data = metadata))$rank,
    design_columns = ncol(model.matrix(~ condition + batch, data = metadata))
  )
})

correction_summaries


Correcting Transcriptomics

Correcting Proteomics

Correcting Metabolomics



modality,features_before,features_after,samples,input_missing_cells,corrected_missing_cells,corrected_rows_with_any_na,batches,design_rank,design_columns
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
Transcriptomics,26907,26907,48,0,0,0,4,7,7
Proteomics,2539,2539,48,0,0,0,4,7,7
Metabolomics,71,71,48,0,0,0,4,7,7


## Save summaries

In [5]:
diagnostic_summaries_df <- bind_rows(diagnostic_summaries)

write.table(correction_summaries, file = file.path(AFTER_DIR, "central_correction_summary.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
write.table(diagnostic_summaries_df, file = file.path(AFTER_DIR, "diagnostic_plot_summary.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

correction_summaries
diagnostic_summaries_df

modality,features_before,features_after,samples,input_missing_cells,corrected_missing_cells,corrected_rows_with_any_na,batches,design_rank,design_columns
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
Transcriptomics,26907,26907,48,0,0,0,4,7,7
Proteomics,2539,2539,48,0,0,0,4,7,7
Metabolomics,71,71,48,0,0,0,4,7,7


modality,state,pca_features
<chr>,<chr>,<int>
Transcriptomics,before,26907
Transcriptomics,after,26907
Proteomics,before,2539
Proteomics,after,2539
Metabolomics,before,71
Metabolomics,after,71


## Session information

In [6]:
sessionInfo()

R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.4 LTS

Matrix products: default
BLAS/LAPACK: /home/yuliya-cosybio/miniforge3/envs/fedRBE/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] viridis_0.6.5     viridisLite_0.4.3 ggsci_4.2.0       umap_0.2.10.0    
 [5] patchwork_1.3.2   gridExtra_2.3     limma_3.66.0      lubridate_1.9.5  
 [9] forcats_1.0.1     stringr_1.6.0     dplyr_1.2.1     